In [15]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [16]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *


spark = (
    SparkSession
        .builder
        .appName("deltaLakeApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
)

sc= spark.sparkContext
spark

In [18]:
yellowTaxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PULocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])

yellowTaxisDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)

yellowTaxisDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [53]:
spark.sql("CREATE DATABASE IF NOT EXISTS TaxisDB")

DataFrame[]

In [25]:
yellowTaxisDF.write.format("parquet").mode("overwrite").partitionBy("VendorID").option("path", "./Files/Output/YellowTaxis.parquet").saveAsTable("TaxisDB.YellowTaxisParquet")

25/07/02 21:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/02 21:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/07/02 21:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/07/02 21:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/07/02 21:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
25/07/02 21:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/07/02 21:27:17 WARN MemoryManager: Total allocation exceeds 95.

In [56]:
# If already executed this step and want to run it again, you can run the next line
# spark.sql("DROP DATABASE IF EXISTS TaxisDB CASCADE")

from delta import *

yellowTaxisDF.write.format("delta").mode("overwrite").partitionBy("VendorID").option("path", "/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxis.delta").saveAsTable("TaxisDB.YellowTaxis")

25/07/02 21:51:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/07/02 21:51:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
25/07/02 21:51:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
25/07/02 21:51:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/07/02 21:51:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
25/07/02 21:51:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
25/07/02 21:51:28 WARN MemoryManager: Total allocation exceeds 95.

In [57]:
# Show audit logs
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------

In [63]:
spark.sql("DESCRIBE TABLE EXTENDED TaxisDB.YellowTaxis").show(100, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                     |comment|
+----------------------------+----------------------------------------------------------------------------------------------+-------+
|VendorID                    |int                                                                                           |NULL   |
|tpep_pickup_datetime        |timestamp                                                                                     |NULL   |
|tpep_dropoff_datetime       |timestamp                                                                                     |NULL   |
|passenger_count             |double                                                                                        |NULL   |
|trip_distance               |double                          

In [72]:
spark.sql("SELECT * FROM TaxisDB.YellowTaxis WHERE VendorID = 3").show(1, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|3       |2022-10-01 12:00:00 |2022-10-01 12:30:00  |1.0            |1.8          |1.0       |N                 |236         |162         |3           |20.0       |0.5  |0.5    |3.0      

In [71]:
spark.sql("INSERT INTO TaxisDB.YellowTaxis (VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, airport_fee) VALUES (3, '2022-10-01 12:00:00', '2022-10-01 12:30:00', 1.0, 1.8, 1.0, 'N', 236, 162, 3, 20.0, 0.5, 0.5, 3.0, 2.5, 0.3, 26.8, 2.5, 0.0)").show()

++
||
++
++



In [ ]:
spark.sql("SELECT * FROM TaxisDB.YellowTaxis WHERE VendorID = 3").show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|3       |2022-10-01 12:00:00 |2022-10-01 12:30:00  |1.0            |1.8          |1.0       |N                 |236         |162         |3           |20.0       |0.5  |0.5    |3.0      

In [74]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------

In [75]:
yellowTaxisAppendDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_append.csv"
)
yellowTaxisAppendDF.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|3       |2022-10-01 05:33:41 |2022-10-01 05:48:39  |1.0            |1.7          |1.0       |N                 |249         |107         |1           |9.5        |3.0  |0.5    |2.65     

In [ ]:
# Append data to the Delta table
yellowTaxisAppendDF.write.format("delta").mode("append").partitionBy("VendorID").save("/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxis.delta")

In [81]:
spark.sql("SELECT * FROM TaxisDB.YellowTaxis WHERE VendorID = 3").show(10, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|3       |2022-10-01 05:33:41 |2022-10-01 05:48:39  |1.0            |1.7          |1.0       |N                 |249         |107         |1           |9.5        |3.0  |0.5    |2.65     

In [78]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                      |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------

In [82]:
# Check files holding the data for VendorID = 3
spark.sql("SELECT INPUT_FILE_NAME(), VendorID, PULocationID, DOLocationID FROM TaxisDB.YellowTaxis WHERE VendorID = 3").show(10, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+------------+------------+
|input_file_name()                                                                                                                                                            |VendorID|PULocationID|DOLocationID|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+------------+------------+
|file:/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxis.delta/VendorID=3/part-00000-79d8e917-c576-4474-9149-572b256b4fe4.c000.snappy.parquet|3       |249         |107         |
|file:/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxis.delta/VendorID=3/part-00000-79d8e917-c576-4474-9149-572b256b4fe4.c

In [83]:
spark.sql("SELECT VendorID, PULocationID, passenger_count FROM TaxisDB.YellowTaxis WHERE VendorID = 3 AND PULocationID = 249").show(10, truncate=False)

+--------+------------+---------------+
|VendorID|PULocationID|passenger_count|
+--------+------------+---------------+
|3       |249         |1.0            |
+--------+------------+---------------+



In [84]:
# Run update statement to change passenger_count to 2.0 for VendorID = 3 and PULocationID = 249
spark.sql("""
UPDATE TaxisDB.YellowTaxis
SET passenger_count = 2.0
WHERE VendorID = 3 AND PULocationID = 249
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



In [85]:
spark.sql("SELECT VendorID, PULocationID, passenger_count FROM TaxisDB.YellowTaxis WHERE VendorID = 3 AND PULocationID = 249").show(10, truncate=False)

+--------+------------+---------------+
|VendorID|PULocationID|passenger_count|
+--------+------------+---------------+
|3       |249         |2.0            |
+--------+------------+---------------+



In [86]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                               

In [87]:
spark.sql("SELECT VendorID, PULocationID, passenger_count FROM TaxisDB.YellowTaxis WHERE VendorID = 3 AND PULocationID = 151").show(10, truncate=False)

+--------+------------+---------------+
|VendorID|PULocationID|passenger_count|
+--------+------------+---------------+
|3       |151         |2.0            |
+--------+------------+---------------+



In [88]:
# Delete rows with VendorID = 3 and PULocationID = 151
spark.sql("""
DELETE FROM TaxisDB.YellowTaxis
WHERE VendorID = 3 AND PULocationID = 151
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



In [89]:
spark.sql("SELECT VendorID, PULocationID, passenger_count FROM TaxisDB.YellowTaxis WHERE VendorID = 3 AND PULocationID = 151").show(10, truncate=False)

+--------+------------+---------------+
|VendorID|PULocationID|passenger_count|
+--------+------------+---------------+
+--------+------------+---------------+



In [90]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                     |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                                                               

In [91]:
# Extract changed records from storage
yellowTaxisChangesDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv("./Files/YellowTaxis_changes.csv")
yellowTaxisChangesDF.show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|3       |2022-10-01 05:33:41 |2022-10-01 05:48:39  |1.0            |1.7          |1.0       |N                 |249         |107         |3           |9.5        |3.0  |0.5    |2.65     

In [92]:
yellowTaxisChangesDF.createOrReplaceTempView("YellowTaxisChanges")

In [93]:
spark.sql("SELECT * FROM YellowTaxisChanges").show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|3       |2022-10-01 05:33:41 |2022-10-01 05:48:39  |1.0            |1.7          |1.0       |N                 |249         |107         |3           |9.5        |3.0  |0.5    |2.65     

In [98]:
spark.sql("""
MERGE INTO TaxisDB.YellowTaxis AS target
USING YellowTaxisChanges AS source
ON target.VendorID = source.VendorID AND target.PULocationID = source.PULocationID
WHEN MATCHED THEN
    UPDATE SET payment_type = source.payment_type
WHEN NOT MATCHED
    AND tpep_pickup_datetime >= '2022-10-01'
    THEN
        INSERT (VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID,
            DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, airport_fee)
        VALUES (source.VendorID, source.tpep_pickup_datetime, source.tpep_dropoff_datetime, source.passenger_count, source.trip_distance, source.RatecodeID, source.store_and_fwd_flag, source.PULocationID,
            source.DOLocationID, source.payment_type, source.fare_amount, source.extra, source.mta_tax, source.tip_amount, source.tolls_amount, source.improvement_surcharge, source.total_amount, source.congestion_surcharge, source.airport_fee)
""").show()

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                4|               2|               0|                2|
+-----------------+----------------+----------------+-----------------+



In [99]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [102]:
# NOT NULL constraint
spark.sql("""
ALTER TABLE TaxisDB.YellowTaxis
          CHANGE COLUMN PULocationID DROP NOT NULL
""").show()

++
||
++
++



In [103]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [105]:
spark.sql("""
ALTER TABLE TaxisDB.YellowTaxis
          CHANGE COLUMN tip_amount SET NOT NULL
""").show()

AnalysisException: Cannot change nullable column to non-nullable: tip_amount.; line 2 pos 0;
AlterColumn resolvedfieldname(StructField(tip_amount,DoubleType,true)), false
+- ResolvedTable org.apache.spark.sql.delta.catalog.DeltaCatalog@563e15f9, TaxisDB.YellowTaxis, DeltaTableV2(org.apache.spark.sql.SparkSession@426244c8,file:/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxis.delta,Some(CatalogTable(
Catalog: spark_catalog
Database: taxisdb
Table: yellowtaxis
Created Time: Wed Jul 02 21:51:25 IST 2025
Last Access: UNKNOWN
Created By: Spark 
Type: EXTERNAL
Provider: delta
Table Properties: [delta.lastCommitTimestamp=1751473289880, delta.lastUpdateVersion=1, delta.minReaderVersion=1, delta.minWriterVersion=2]
Location: file:/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxis.delta
Partition Columns: [`VendorID`]
Schema: root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |--

In [106]:
spark.sql("""ALTER TABLE TaxisDB.YellowTaxis
          ADD CONSTRAINT passenger_count CHECK (passenger_count < 5
          )""").show()

AnalysisException: [DELTA_NEW_CHECK_CONSTRAINT_VIOLATION] 227699 rows in spark_catalog.taxisdb.yellowtaxis violate the new CHECK constraint (passenger_count < 5)

In [108]:
spark.sql("""ALTER TABLE TaxisDB.YellowTaxis
          ADD CONSTRAINT PassengerCheck CHECK (passenger_count <= 9 OR passenger_count IS NULL
          )""").show()

++
||
++
++



In [109]:
spark.sql("INSERT INTO TaxisDB.YellowTaxis (VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount, congestion_surcharge, airport_fee) VALUES (3, '2022-10-01 12:00:00', '2022-10-01 12:30:00', 10.0, 1.8, 1.0, 'N', 236, 162, 3, 20.0, 0.5, 0.5, 3.0, 2.5, 0.3, 26.8, 2.5, 0.0)").show()

25/07/02 23:20:14 ERROR Executor: Exception in task 0.0 in stage 268.0 (TID 2919)
org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint passengercheck ((passenger_count <= 9) OR (passenger_count IS NULL)) violated by row with values:
 - passenger_count : 10.0
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:75)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:101)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:112)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.delta.constraints.DeltaInvariantChecker

Py4JJavaError: An error occurred while calling o34.sql.
: org.apache.spark.sql.delta.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint passengercheck ((passenger_count <= 9) OR (passenger_count IS NULL)) violated by row with values:
 - passenger_count : 10.0
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:75)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:101)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:112)
	at org.apache.spark.sql.delta.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.delta.constraints.DeltaInvariantCheckerExec.$anonfun$doExecute$

In [110]:
spark.sql("""ALTER TABLE TaxisDB.YellowTaxis
          DROP CONSTRAINT PassengerCheck
          """).show()

++
||
++
++



In [111]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [112]:
# Check passenger_count before update
spark.sql("SELECT passenger_count FROM TaxisDB.YellowTaxis WHERE VendorID = 3 AND PULocationID = 1").show(10, truncate=False)

+---------------+
|passenger_count|
+---------------+
|0.0            |
+---------------+



In [113]:
# Update passenger_count to 2.0 for VendorID = 3 and PULocationID = 1
spark.sql("""UPDATE TaxisDB.YellowTaxis
SET passenger_count = 2.0
WHERE VendorID = 3 AND PULocationID = 1
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



In [114]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [115]:
# Check at initial version
spark.sql("SELECT passenger_count FROM TaxisDB.YellowTaxis VERSION AS OF 0 WHERE VendorID = 3 AND PULocationID = 1").show(10, truncate=False)

+---------------+
|passenger_count|
+---------------+
+---------------+



In [117]:
spark.sql("SELECT passenger_count FROM TaxisDB.YellowTaxis VERSION AS OF 9 WHERE VendorID = 3 AND PULocationID = 1").show(10, truncate=False)

+---------------+
|passenger_count|
+---------------+
|0.0            |
+---------------+



In [118]:
spark.sql("SELECT passenger_count FROM TaxisDB.YellowTaxis TIMESTAMP AS OF '2025-07-02 23:27:00' WHERE VendorID = 3 AND PULocationID = 1").show(10, truncate=False)

+---------------+
|passenger_count|
+---------------+
|0.0            |
+---------------+



In [119]:
# Restore table to earlier version
spark.sql("RESTORE TABLE TaxisDB.YellowTaxis TO VERSION AS OF 9").show()

25/07/02 23:35:32 WARN DAGScheduler: Broadcasting large task binary with size 1073.8 KiB


+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|table_size_after_restore|num_of_files_after_restore|num_removed_files|num_restored_files|removed_files_size|restored_files_size|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|                81922831|                        27|                1|                 1|              5728|               5727|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+



In [120]:
spark.sql("DESCRIBE HISTORY TaxisDB.YellowTaxis").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [121]:
spark.sql("SELECT passenger_count FROM TaxisDB.YellowTaxis WHERE VendorID = 3 AND PULocationID = 1").show(10, truncate=False)

+---------------+
|passenger_count|
+---------------+
|0.0            |
+---------------+

